<a href="https://colab.research.google.com/github/chandrachrist/guessing-game/blob/master/CI_CD_for_Machine_Learning_Systems_demo_U4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CI/CD for Machine Learning Systems
This notebook demonstrates key components of MLOps CI/CD:
1. **Version Control** for Code (Git), Data, and Models (DVC/MLflow).
2. **Automated Testing** of Data and ML pipelines.

## 1. Environment Setup & Dependencies
First, we install standard ML packages along with libraries commonly used in CI/CD pipelines such as `pytest` (for automated testing) and `mlflow` (for tracking and versioning models).

In [ ]:
!pip install -q pytest mlflow scikit-learn pandas

## 2. Automated Pipeline Testing (Unit Testing with pytest)
In a CI/CD environment, code is verified automatically. We'll write a simple data pre-processing function and then write a test file using `pytest` to validate it automatically.

In [ ]:
%%writefile preprocess.py
import pandas as pd

def clean_data(df):
    # Drop rows where target variable 'label' is null
    if 'label' in df.columns:
        df = df.dropna(subset=['label'])
    # Fill missing feature values with 0
    df = df.fillna(0)
    return df

Writing preprocess.py


In [ ]:
%%writefile test_preprocess.py
import pandas as pd
import numpy as np
from preprocess import clean_data

def test_clean_data():
    # Arrange: Create dummy dirty dataframe
    raw_data = pd.DataFrame({
        'feature1': [1.0, np.nan, 3.0],
        'label': [1, 0, np.nan]
    })

    # Act
    cleaned = clean_data(raw_data)

    # Assert
    assert len(cleaned) == 2, "Should have dropped rows with missing labels"
    assert cleaned['feature1'].isna().sum() == 0, "Should have filled missing features with 0"

Writing test_preprocess.py


Now run the test suite as it would be run inside a GitHub Actions runner or GitLab CI pipeline:

In [ ]:
!pytest test_preprocess.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 1 item                                                               

test_preprocess.py .                                                     [100%]

============================== 1 passed in 1.57s ===============================


### Simulating a Pipeline/Test Failure
To show students how a CI/CD runner catches code bugs automatically, we will intentionally modify our unit test file (`test_preprocess.py`) to assert an incorrect value (`3` instead of the expected `2` rows).

In [ ]:
%%writefile test_preprocess.py
import pandas as pd
import numpy as np
from preprocess import clean_data

def test_clean_data():
    # Arrange: Create dummy dirty dataframe
    raw_data = pd.DataFrame({
        'feature1': [1.0, np.nan, 3.0],
        'label': [1, 0, np.nan]
    })

    # Act
    cleaned = clean_data(raw_data)

    # Assert - Repaired assertion back to 2 rows
    assert len(cleaned) == 2, "Should have dropped rows with missing labels"
    assert cleaned['feature1'].isna().sum() == 0, "Should have filled missing features with 0"

Writing test_preprocess.py


In [ ]:
%%writefile preprocess.py
import pandas as pd

def clean_data(df):
    # Drop rows where target variable 'label' is null
    if 'label' in df.columns:
        df = df.dropna(subset=['label'])
    # Fill missing feature values with 0
    df = df.fillna(0)
    return df

Overwriting preprocess.py


In [ ]:
!pytest test_preprocess.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 1 item                                                               

test_preprocess.py .                                                     [100%]

============================== 1 passed in 0.67s ===============================


## 3. Version Control for Models with MLflow
To ensure repeatability, every trained model artifact needs versioning. We can use MLflow to log parameters, metrics, and save/register the model version.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Create dummy dataset
X, y = make_classification(n_samples=100, n_features=4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Configure MLflow experiment
mlflow.set_experiment("Classification_Pipeline")

with mlflow.start_run() as run:
    # Define parameters
    n_estimators = 100
    max_depth = 3

    # Log parameters
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)

    # Train
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)

    # Log metric
    acc = model.score(X_test, y_test)
    mlflow.log_metric("accuracy", acc)

    # Version Control the Model Artifact - passing trusted types to pass skops validation
    mlflow.sklearn.log_model(
        model,
        "random-forest-model",
        skops_trusted_types=["sklearn.tree._tree.Tree"]
    )

    print(f"Run tracked successfully. Run ID: {run.info.run_id}")
    print(f"Model Accuracy: {acc}")

2026/11/07 03:16:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run tracked successfully. Run ID: 43b116d8960b44a38cece5ebc80587f4
Model Accuracy: 0.95
